# Deploy pipelines

**Normal workflow: run this notebook, then notebook 05.**
You do not also need notebooks 01 and 02.

Run this notebook after cloning the complete application. It uses a complete
local config without reading Secrets Manager. For an incomplete config, it
loads the account's shared settings. It prepares sample data only when no
dataset is configured and publishes the training and inference pipelines.

On later runs it validates and reuses the configured inputs. Existing
namespace and data choices are retained. Partial or inaccessible inputs
stop deployment rather than triggering replacement data.

The shared secret is optional when local config is complete. To support fresh
workspaces, an operator can create `wsf-mlops-setup` with
`operations/create_setup_secret.ipynb` once per account/region. Incomplete setup
and explicit `REFRESH_SHARED=True` read that secret and require access to it.
This notebook does not provision the bucket, role, image or Registry groups;
those platform resources must exist.

Notebooks 01 and 02 remain optional detailed preparation/publication tools.
Notebook 05 is separate so you can select models and inputs, launch the test
in the background, and rerun tests without republishing.

The shared data seed is **42**. To reproduce the same input bytes, use the same
metadata, model targets, generation settings, code and dependency versions.
Compare the three published SHA-256 values. Reusing data does not apply a new
seed: select `generate` explicitly if you intentionally change generation settings.

If deployment fails, **save this notebook with its outputs; do not clear them**.
The failing cell prints the stage, AWS target/response context, error chain,
source hashes and original traceback. Share the saved notebook for diagnosis.
The report and detailed stdout/stderr log remain under `.runs/setup/` as well.
A failed run can have published one pipeline before a later operation failed;
the diagnostic progress records which versions were verified.


## Choose deployment options

- Use the current AWS identity and region; complete local config skips Secrets Manager.
- Use `auto` to generate only when no dataset is configured, or `reuse` to require existing data.
- Select `generate` only when you deliberately want a new sample.
- Keep shared-setting refresh off unless you intend to replace conflicting local platform values.


In [ ]:
# - Set deployment options before loading the helpers.
# - Locate this checkout and use the current AWS session; no deployment runs here.
# - Print bootstrap context so import/checkout failures remain in saved notebook output.
SECRET_NAME = "wsf-mlops-setup"  # Used for incomplete config or an explicit refresh.
REGION = None  # Use the AWS region configured in this environment.
DATA_MODE = "auto"
REFRESH_SHARED = False
DATA_SEED = 42
PREPARATION = {"sample_scale": 100, "numeric_zero_fraction": 0.10, "split_seed": DATA_SEED}
PUBLICATION = {}

from pathlib import Path
import sys

ROOT = SOURCE = deployment = session = result = None
try:
    import boto3
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "mlops-models/src").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Open this notebook inside the complete application checkout.")
    SOURCE = ROOT / "notebooks/source/setup"
    sys.path.insert(0, str(SOURCE))
    import deploy_pipelines as deployment
    assert Path(deployment.__file__).resolve() == (SOURCE / "deploy_pipelines.py").resolve(), (
        "Restart the kernel: the deployment helper came from another checkout."
    )
    session = boto3.Session(region_name=REGION)
    print("Checkout:", ROOT, "| Region:", session.region_name)
    print("Deployment helper:", deployment.__file__)
except BaseException as error:
    print("DEPLOYMENT BOOTSTRAP FAILED:", type(error).__name__, str(error))
    print("Working directory:", Path.cwd(), "| Checkout:", ROOT, "| Helper directory:", SOURCE)
    print("Python:", sys.version, "| Executable:", sys.executable, "| Requested region:", REGION)
    print("Save this notebook with its outputs and traceback for diagnosis.")
    raise


## Deploy and return the result

- Validate complete local config, or fill missing shared values from the secret.
- Generate missing data or validate the exact configured files/S3 objects.
- Verify uploads and upsert pipeline definitions using the existing publication flow.
- Return the result below; detailed output and any failure are saved under `.runs/setup/`.


In [ ]:
# - Validate settings and inputs, then publish and verify the pipelines.
# - Keep result empty on failure so an earlier success is never displayed.
# - Print diagnostics and preserve the original traceback if any deployment stage fails.
result = None
result = deployment.deploy_pipelines(
    ROOT, session,
    secret_name=SECRET_NAME,
    refresh_shared=REFRESH_SHARED,
    data_mode=DATA_MODE,
    preparation_options=PREPARATION,
    publication_options=PUBLICATION,
)


## Read the result

- Show elapsed time and five actual training rows; `<missing>` marks blank CSV cells.
- For generated data, show the seed, split sizes and missing-cell counts.
- Print the exact published S3 paths and hashes for comparing inputs between workspaces.
- Keep full diagnostics in the saved report/log; continue with notebook 05 to execute pipelines.


In [ ]:
if result is None:
    raise RuntimeError("Deployment did not complete. Save the earlier diagnostic output and traceback.")

import pandas as pd

print(f"Completed in {result['elapsed_seconds']:.1f}s | Data: {result['data_action']}")
if "preparation" in result:
    prepared = result["preparation"]
    print(f"Seed: {prepared['seed']} | Generation: {prepared['elapsed_seconds']:.1f}s")
    print("Rows: " + " | ".join(f"{kind}: {count:,}" for kind, count in prepared["rows"].items()))
    print("Missing cells: " + " | ".join(f"{kind}: {count:,}" for kind, count in prepared["missing_cells"].items()))
    if any(prepared["missing_cells"].values()):
        print("Source missing values were preserved; column counts are in the saved preparation report.")
if result["preview"]:
    print("\nTraining sample — first 5 rows, first 6 columns (<missing> = blank CSV cell):")
    print(pd.DataFrame(result["preview"]).replace("", "<missing>").to_string(index=False, max_colwidth=24))
print("\nPublished data:")
for kind, key in (("training", "csv_input"), ("testing", "test_csv_input"), ("inference", "inference_csv_input")):
    if result["inputs"].get(key):
        print(f"  {kind}: {result['inputs'][key]}")
        if kind in result["sha256"]:
            print(f"    SHA-256: {result['sha256'][kind]}")
print("\nPipelines: " + " | ".join(item["name"] for item in result["pipelines"].values()))
print("Details:", result["report_path"])
